# OCR seletivo do corpus — documentos bloqueados

Processa somente `AS-CN-01`, `AS-CN-02` e `SG_015`, preservando página e document_id.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-por tesseract-ocr-chi-sim
!pip -q install pymupdf pytesseract pandas pillow


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
CORPUS_DIR = Path('/content/drive/MyDrive/Colab Notebooks/corpus')
OUTPUT_DIR = Path('/content/drive/MyDrive/ragdiretrizes/ocr_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGETS = {
    'AS-CN-01': {'relative_path': 'china/china_information_technology_curriculum_standard_2022.pdf', 'lang': 'chi_sim+eng'},
    'AS-CN-02': {'relative_path': 'china/china_compulsory_education_curriculum_plan_2022.pdf', 'lang': 'chi_sim+eng'},
    'SG_015': {'relative_path': 'singapura/singapura_e_brasil_diretrizes_de_computação.pdf', 'lang': 'por+eng'},
}
for document_id, cfg in TARGETS.items():
    p = CORPUS_DIR / cfg['relative_path']
    print(document_id, 'OK' if p.exists() else 'NÃO ENCONTRADO', p)


In [ ]:
import re, pandas as pd, pymupdf, pytesseract
from PIL import Image
from io import BytesIO
def clean_text(text):
    text=text.replace('\x0c',' ')
    text=re.sub(r'[ \t]+',' ',text)
    text=re.sub(r'\n{3,}','\n\n',text)
    return text.strip()
def metrics(text):
    n=len(text)
    if n==0: return 0,0.0,'EMPTY'
    ratio=sum(ch.isalnum() for ch in text)/n
    status='LOW_TEXT' if n<40 else ('LOW_QUALITY' if ratio<0.25 else 'OK')
    return n,round(ratio,4),status
def ocr_pdf(document_id,pdf_path,lang,dpi=220):
    doc=pymupdf.open(pdf_path)
    parts=[f'# OCR — {document_id}','',f'- source_file: {pdf_path.name}',f'- ocr_language: {lang}',f'- total_pages: {len(doc)}','']
    rows=[]
    for i in range(len(doc)):
        page=doc[i]; page_no=i+1
        pix=page.get_pixmap(dpi=dpi,alpha=False)
        image=Image.open(BytesIO(pix.tobytes('png')))
        text=clean_text(pytesseract.image_to_string(image,lang=lang,config='--oem 1 --psm 6'))
        chars,ratio,status=metrics(text)
        parts += [f'## PAGE {page_no}','',text if text else '[SEM TEXTO OCR]','']
        rows.append({'document_id':document_id,'file_name':pdf_path.name,'page':page_no,'ocr_language':lang,'chars':chars,'alnum_ratio':ratio,'page_status':status})
        print(f'{document_id} | página {page_no}/{len(doc)} | chars={chars} | status={status}')
    doc.close()
    out=OUTPUT_DIR/f'{document_id}.md'
    out.write_text('\n'.join(parts),encoding='utf-8')
    return out,rows


In [ ]:
all_rows=[]
for document_id,cfg in TARGETS.items():
    source=CORPUS_DIR/cfg['relative_path']
    out,rows=ocr_pdf(document_id,source,cfg['lang'])
    all_rows.extend(rows)
    print('Gerado:',out)


In [ ]:
df=pd.DataFrame(all_rows)
pages_csv=OUTPUT_DIR/'ocr_validation_pages.csv'
df.to_csv(pages_csv,index=False,encoding='utf-8-sig')
summary=(df.groupby('document_id').agg(total_pages=('page','count'),pages_ok=('page_status',lambda s:int((s=='OK').sum())),pages_low_text=('page_status',lambda s:int((s=='LOW_TEXT').sum())),pages_low_quality=('page_status',lambda s:int((s=='LOW_QUALITY').sum())),pages_empty=('page_status',lambda s:int((s=='EMPTY').sum())),total_chars=('chars','sum')).reset_index())
summary['percent_pages_ok']=(summary['pages_ok']/summary['total_pages']*100).round(2)
summary['ocr_validation_status']=summary.apply(lambda r:'READY_FOR_REVIEW' if r['pages_empty']==0 and r['percent_pages_ok']>=80 else 'REVIEW_REQUIRED',axis=1)
summary_csv=OUTPUT_DIR/'ocr_validation_summary.csv'
summary.to_csv(summary_csv,index=False,encoding='utf-8-sig')
display(summary)
display(df[df['page_status']!='OK'])
print(pages_csv)
print(summary_csv)


## Saídas esperadas

- `AS-CN-01.md`
- `AS-CN-02.md`
- `SG_015.md`
- `ocr_validation_pages.csv`
- `ocr_validation_summary.csv`

Não substitua os PDFs originais.
